# SpaNCy-Shift CFM Explorer

OT-Conditional Flow Matching as Stage 2 batch correction for CyCIF multiplexed imaging.

**Stage 1 (analytic):** Per-marker shift toward KL-medoid reference → `normalized_base`. kBET ≈ 0.631.

**Stage 2 (OT-CFM):** `FlowMLP` conditioned on source batch learns a velocity field transporting
cells toward the reference-batch distribution. Training: mini-batch OT coupling (Hungarian algorithm
on 256×256 L2 cost matrix) + MSE velocity loss. Inference: Euler ODE integration t=0 → t=1.

No torch-geometric. No adversarial training. No bimodal masking needed — velocity field learns
the full joint distribution including bimodal structure naturally.

**Sections:**
0. Colab Setup
1. Load & Inspect Data
2. Bimodal Marker Detection
2b. Reference Sample Selection
3. Training (Stage 2 OT-CFM)
3b. n_steps Sweep (correction strength calibration)
4. Inference & Histogram Inspection
5. Batch adj-R² Diagnostics
6. Positive Population Preservation Check
7. Histogram Comparison PDF
8. kBET Evaluation

## 0. Colab Setup

Run first on Google Colab. Installs dependencies and uploads both `.py` files.

**Note:** No torch-geometric needed for CFM — pure torch + scipy.

In [ ]:
!pip install -q anndata scanpy pegasuspy pegasusio
print('Done.')

In [ ]:
# Upload BOTH spancy_shift.py AND spancy_shift_cfm.py
from google.colab import files
uploaded = files.upload()

import os
assert os.path.exists('spancy_shift.py'), 'spancy_shift.py not found'
assert os.path.exists('spancy_shift_cfm.py'), 'spancy_shift_cfm.py not found'
print(f"spancy_shift.py    ({os.path.getsize('spancy_shift.py'):,} bytes)")
print(f"spancy_shift_cfm.py ({os.path.getsize('spancy_shift_cfm.py'):,} bytes)")

In [ ]:
import importlib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

import spancy_shift
import spancy_shift_cfm
from spancy_shift import (
    load_adata, log1p_scale,
    detect_bimodal_markers, find_best_sample_per_marker,
    shift_normalize_per_marker,
    positive_population_table, summarize_positive_population,
    per_marker_batch_r2,
)
from spancy_shift_cfm import (
    FlowMLP, BatchBalancedSampler, ot_couple,
    train_cfm, normalize_adata_cfm,
)

# After re-uploading files:
# importlib.reload(spancy_shift); importlib.reload(spancy_shift_cfm)
# from spancy_shift import *; from spancy_shift_cfm import *

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# Download PRAD dataset
!wget -q https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad
print('Download complete.')

## 1. Load & Inspect Data

In [ ]:
DATA_PATH = 'PRAD_anndata.h5ad'

adata = load_adata(DATA_PATH)
print(adata)
print(f"\nMarkers ({adata.n_vars}): {list(adata.var_names)}")
print(f"Batches:  {sorted(adata.obs['batch_id'].unique().tolist())}")
print(f"Samples:  {sorted(adata.obs['sample_id'].unique().tolist())}")
print(f"\nobs columns: {list(adata.obs.columns)}")

marker_names = list(adata.var_names)
X_raw = np.asarray(adata.X.toarray() if sp.issparse(adata.X) else adata.X)
batch_vals = adata.obs['batch_id'].values
unique_batches = sorted(adata.obs['batch_id'].unique())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
axes[0].boxplot([X_raw[:, i] for i in range(X_raw.shape[1])], labels=marker_names, vert=True)
axes[0].set_xticklabels(marker_names, rotation=90, fontsize=8)
axes[0].set_title('Raw marker distributions (all cells)')
axes[0].set_ylabel('Intensity')
for b in unique_batches:
    mask = batch_vals == b
    axes[1].plot(X_raw[mask].mean(axis=0), marker='o', markersize=3, label=str(b), alpha=0.8)
axes[1].set_xticks(range(len(marker_names)))
axes[1].set_xticklabels(marker_names, rotation=90, fontsize=8)
axes[1].set_title('Per-batch mean intensity')
axes[1].legend(fontsize=6, ncol=2)
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Bimodal Marker Detection (preview on raw data)

Same as spancy_shift — CFM does NOT need explicit bimodal masking.
The velocity field learns the full joint distribution including bimodal structure.
Shown here for reference only.

In [ ]:
BIMODAL_MIN_BATCH_FRAC = 0.5

X_log_preview = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)
batch_codes_preview = adata.obs['batch_id'].astype('category').cat.codes.values.astype('int64')

marker_is_bimodal, thresholds = detect_bimodal_markers(
    X_log_preview, marker_names,
    batch_codes=batch_codes_preview,
    min_prominence_frac=0.05,
    bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
)

bimodal_names = [marker_names[i] for i in range(len(marker_names)) if marker_is_bimodal[i]]

print(f"Bimodal markers ({marker_is_bimodal.sum()}):")
for i, name in enumerate(marker_names):
    if marker_is_bimodal[i]:
        print(f"  {name:>12s}  threshold={thresholds[i]:.3f}")
print(f"\nUnimodal markers ({(~marker_is_bimodal).sum()}):")
for i, name in enumerate(marker_names):
    if not marker_is_bimodal[i]:
        print(f"  {name}")
print("\nNote: CFM learns the full distribution — no special handling for bimodal markers.")

## 2b. Reference Sample Selection

For each marker, find the medoid sample (lowest mean pairwise KL divergence).
The OT coupling targets cells from whichever batch contains the most reference samples.

In [ ]:
ref_sample_per_marker = find_best_sample_per_marker(adata)

ref_df = pd.DataFrame(
    [(m, s) for m, s in ref_sample_per_marker.items()],
    columns=['marker', 'reference_sample'],
)
print('Per-marker reference sample:\n')
print(ref_df.to_string(index=False))

ref_counts = ref_df['reference_sample'].value_counts()
print(f'\nDistinct reference samples: {len(ref_counts)}')
print(ref_counts.to_string())

fig, ax = plt.subplots(figsize=(14, 3))
sample_order = sorted(adata.obs['sample_id'].unique())
ref_matrix = [sum(v == s for v in ref_sample_per_marker.values()) for s in sample_order]
ax.bar(range(len(sample_order)), ref_matrix, color='steelblue', alpha=0.85)
ax.set_xticks(range(len(sample_order)))
ax.set_xticklabels(sample_order, rotation=45, ha='right')
ax.set_ylabel('# markers using as reference')
ax.set_title('Reference sample selection (medoid)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## 3. Stage 2 Training (OT-CFM)

Trains `FlowMLP` on Stage 1 output (`normalized_base`).

**Inside `train_cfm()`:**
1. Runs `shift_normalize_per_marker()` → `X_base` (Stage 1)
2. Fits `RobustScaler` on `log1p(X_base)` → `X_scaled`
3. Identifies reference batch (majority vote over per-marker reference samples)
4. For each training step:
   - Sample source cells from all batches (batch-balanced)
   - OT coupling: Hungarian algorithm on 256×256 L2 cost matrix → phenotype-matched pairs
   - Interpolate at random t ∈ [0,1]: `x_t = (1-t)*x_0 + t*x_1 + σ*noise`
   - Predict velocity toward x_1; target = `x_1 - x_0`; loss = MSE

**Inference:** Euler ODE from t=0 to t=1 (n_steps Euler steps).
`n_steps` controls correction strength: 5 = very conservative, 50 = full transport.

In [ ]:
N_STEPS = 20    # Euler ODE steps at inference — tune in sec3b
N_EPOCHS = 10   # quick test; use 50-100 for production

model, trained_scaler, trained_ref, history = train_cfm(
    adata,
    n_epochs=N_EPOCHS,
    lr=1e-3,
    device_str=DEVICE,
    warmup_epochs=2,
    n_per_batch=256,
    ot_samples=256,
    sigma_min=0.01,
    hidden=512,
    n_layers=6,
    bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
    ref_sample_per_marker=ref_sample_per_marker,
)

print(f'\nTraining complete.')
print(f'  Reference batch: {model._ref_batch_name} (code {model._ref_batch_code})')
print(f'  Final loss={history["loss"][-1]:.6f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history['loss'], color='k', linewidth=1.5)
axes[0].set_title('OT-CFM Velocity Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['lr'], color='purple', linewidth=1.5)
axes[1].set_title('Learning Rate')
axes[1].set_xlabel('Epoch')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Stage 2 Training — FlowMLP (OT-CFM)', fontsize=13)
plt.tight_layout()
plt.show()

## 3b. n_steps Sweep — Correction Strength Calibration

OT-CFM's correction strength is controlled by `n_steps` (number of Euler integration steps).
Fewer steps = cells moved less = more conservative = better biology preservation.
More steps = cells moved more = stronger alignment = better kBET.

This sweep helps find the right balance before running full kBET evaluation.
Check ECAD histogram — it must remain bimodal at the chosen n_steps.

In [ ]:
N_STEPS_SWEEP = [5, 20, 50]   # conservative → aggressive
BIOLOGY_MARKER = 'ECAD'       # bimodal marker — must stay bimodal
CHECK_MARKERS = ['ECAD', 'CD45', 'ChromA']  # markers to inspect

results_sweep = {}
for n_steps_val in N_STEPS_SWEEP:
    print(f'\n--- n_steps={n_steps_val} ---')
    adata_sw = normalize_adata_cfm(
        adata, model, trained_scaler, trained_ref,
        n_steps=n_steps_val,
        inference_batch_size=4096,
        device_str=DEVICE,
        layer_name=f'norm_nsteps{n_steps_val}',
        keep_base_layer=True,
    )
    results_sweep[n_steps_val] = np.asarray(adata_sw.layers[f'norm_nsteps{n_steps_val}'])
print('\nSweep done.')

In [ ]:
# Histogram comparison across n_steps values
n_rows = len(CHECK_MARKERS)
n_cols = len(N_STEPS_SWEEP) + 1  # +1 for Stage 1 baseline
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))

for row, mname in enumerate(CHECK_MARKERS):
    if mname not in marker_names:
        continue
    m_idx = marker_names.index(mname)

    for col, (label, data) in enumerate(
        [('Stage 1 (base)', np.asarray(adata_sw.layers['normalized_base']))]
        + [(f'CFM n_steps={ns}', results_sweep[ns]) for ns in N_STEPS_SWEEP]
    ):
        ax = axes[row, col]
        for b in unique_batches:
            mask = batch_vals == b
            vals = np.log1p(np.clip(data[mask, m_idx], 0, None))
            ax.hist(vals, bins=80, alpha=0.4, density=True, label=str(b))
        ax.set_title(f'{mname} — {label}', fontsize=9)
        ax.set_xlabel('log1p')
        if col == 0:
            ax.set_ylabel('Density')
        if row == 0 and col == 0:
            ax.legend(fontsize=6, ncol=2, title='Batch')

plt.suptitle(f'n_steps sweep — key markers', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nChoose n_steps where ECAD remains bimodal and batches overlap visually.')
print(f'Recommended starting point: N_STEPS = 20')

## 4. Inference & Output Inspection

Run full inference with chosen `N_STEPS`. Produces both Stage 1 and Stage 2 layers.

In [ ]:
# Run inference with the chosen N_STEPS
adata_norm = normalize_adata_cfm(
    adata, model, trained_scaler, trained_ref,
    n_steps=N_STEPS,
    inference_batch_size=4096,
    device_str=DEVICE,
    layer_name='normalized',
    keep_base_layer=True,
)

X_base = np.asarray(adata_norm.layers['normalized_base'])
X_norm = np.asarray(adata_norm.layers['normalized'])

print(f'Stage 1 (normalized_base): min={X_base.min():.4f}  max={X_base.max():.4f}  mean={X_base.mean():.4f}')
print(f'Stage 2 (normalized):      min={X_norm.min():.4f}  max={X_norm.max():.4f}  mean={X_norm.mean():.4f}')
print(f'\nDelta stats (Stage 2 - Stage 1):')
delta_arr = X_norm - X_base
for i, mname in enumerate(marker_names):
    print(f'  {mname:>12s}  mean_delta={delta_arr[:, i].mean():.4f}  std_delta={delta_arr[:, i].std():.4f}')

In [ ]:
# Compare raw vs Stage 1 vs Stage 2 for unimodal markers
# Although CFM corrects the full joint distribution (no masking), bimodal markers can be
# checked in the n_steps sweep (sec3b) for bimodality preservation. Here focus on unimodal
# markers where Stage 2 batch alignment is the primary objective.
unimodal_names = [marker_names[i] for i in range(len(marker_names)) if not marker_is_bimodal[i]]
plot_markers = unimodal_names[:4] if len(unimodal_names) >= 4 else unimodal_names

layers_to_show = [
    (None,             'Raw'),
    ('normalized_base', 'Stage 1 (analytic)'),
    ('normalized',      'Stage 2 (OT-CFM)'),
]

fig, axes = plt.subplots(len(plot_markers), len(layers_to_show),
                          figsize=(6 * len(layers_to_show), 4 * len(plot_markers)))
if len(plot_markers) == 1:
    axes = axes.reshape(1, -1)

for row, mname in enumerate(plot_markers):
    m_idx = marker_names.index(mname)
    for col, (layer_key, title) in enumerate(layers_to_show):
        ax = axes[row, col]
        data = X_raw if layer_key is None else np.asarray(adata_norm.layers[layer_key])
        for b in unique_batches:
            mask = batch_vals == b
            vals = np.log1p(np.clip(data[mask, m_idx], 0, None))
            ax.hist(vals, bins=80, alpha=0.4, density=True, label=str(b))
        ax.set_title(f'{mname} — {title}')
        ax.set_xlabel('log1p intensity')
        if col == 0:
            ax.set_ylabel('Density')
        if row == 0 and col == 0:
            ax.legend(fontsize=6, title='Batch', ncol=2)

plt.suptitle(f'Unimodal Marker Distributions: Raw → Stage 1 → Stage 2 (CFM, n_steps={N_STEPS})\n'
             '(bimodal marker preservation checked in sec3b n_steps sweep)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Batch adj-R² Diagnostics

Adjusted R² for each marker regressed on batch labels. Lower = batch effect better removed.

In [ ]:
batch_arr = adata_norm.obs['batch_id'].values
X_base_arr = np.asarray(adata_norm.layers['normalized_base'])
X_norm_arr = np.asarray(adata_norm.layers['normalized'])

r2_raw   = per_marker_batch_r2(np.log1p(np.clip(X_raw, 0, None)),       batch_arr, marker_names)
r2_base  = per_marker_batch_r2(np.log1p(np.clip(X_base_arr, 0, None)),  batch_arr, marker_names)
r2_cfm   = per_marker_batch_r2(np.log1p(np.clip(X_norm_arr, 0, None)),  batch_arr, marker_names)

r2_compare = (
    r2_raw .rename(columns={'adj_r2': 'raw'})
    .merge(r2_base.rename(columns={'adj_r2': 'stage1_base'}), on='marker')
    .merge(r2_cfm .rename(columns={'adj_r2': 'stage2_cfm'}),  on='marker')
)

print('Per-Marker Batch adj-R² (lower = better):\n')
print(r2_compare.to_string(index=False, float_format='%.4f'))
print(f'\nMean adj-R²:')
for col in ['raw', 'stage1_base', 'stage2_cfm']:
    print(f'  {col:>14s}: {r2_compare[col].mean():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
x = np.arange(len(r2_compare))
w = 0.25
ax.bar(x - w, r2_compare['raw'],          w, label='Raw',            color='salmon',      alpha=0.85)
ax.bar(x,     r2_compare['stage1_base'],  w, label='Stage 1 (base)', color='steelblue',   alpha=0.85)
ax.bar(x + w, r2_compare['stage2_cfm'],   w, label='Stage 2 (CFM)',  color='forestgreen', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(r2_compare['marker'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Adjusted R²')
ax.set_title('Per-Marker Batch adj-R² — lower is better')
ax.axhline(0.05, color='red', linestyle='--', alpha=0.5, label='Target (0.05)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Positive Population Preservation Check

% positive cells per marker per sample using **per-sample GMM** (2-component Gaussian mixture
fitted on each sample's raw data in log10 space; threshold applied to both raw and normalized
for that sample). Delta = normalized − raw. Target: |delta| < 5% per marker.

**Headline metric**: the full per-marker mean Δ (± SD) table over all 20 markers, sorted by
mean Δ, with a `pass_5pct` column and a count of markers within ±5%. This is the primary
biology-preservation result. A high SD (CD45, CDX2, DAPI_R1, ...) flags a unimodal marker whose
GMM threshold sits near the histogram peak — its large |Δ| is threshold instability, not real
biology distortion. Read mean Δ alongside its SD; do **not** drop markers.

The density-at-threshold reliability filter (`summarize_positive_population()`) is kept **for
reference only** at the end of the cell — it drops most unimodal markers (often leaving ~3),
too few to compare methods, so it is not the primary metric.

In [ ]:
pop_table_base = positive_population_table(
    adata_norm, norm_layer='normalized_base', sample_col='sample_id'
)
pop_table = positive_population_table(
    adata_norm, norm_layer='normalized', sample_col='sample_id'
)

# ── HEADLINE: full per-marker table, ALL 20 markers (primary biology metric) ──
# mean Δ = mean over samples of (norm − raw) % positive. SD = inter-sample spread.
# High SD (CD45, CDX2, DAPI_R1, ...) flags a unimodal marker whose GMM threshold sits
# near the histogram peak → the large |Δ| reflects threshold instability, NOT real
# biology distortion. Read mean Δ alongside its SD; do not filter markers out.
def pop_summary(pt):
    s = pt.groupby('marker').agg(
        mean_pct_raw=('pct_pos_raw', 'mean'),
        mean_pct_norm=('pct_pos_norm', 'mean'),
        mean_delta=('delta', 'mean'),
        sd_delta=('delta', 'std'),
        mean_abs_delta=('delta', lambda x: x.abs().mean()),
    ).round(2)
    s['pass_5pct'] = s['mean_delta'].abs() < 5
    return s.sort_values('mean_delta')

for label, pt in [('Stage 1 (base)', pop_table_base), ('Stage 2 CFM (norm)', pop_table)]:
    summary = pop_summary(pt)
    n_pass = int(summary['pass_5pct'].sum())
    print(f'Positive Population Preservation — {label} vs raw  (ALL 20 markers):')
    print('=' * 95)
    print(summary.to_string())
    print(f'\n  Markers within ±5% mean Δ: {n_pass}/{len(summary)}')
    print(f'  Overall mean |Δ|: {pt["delta"].abs().mean():.2f}%   max |Δ|: {pt["delta"].abs().max():.2f}%')
    print()

print('Target: |mean Δ| < 5% per marker. High-SD failing markers are unimodal (threshold')
print('near the histogram peak) — interpret their |Δ| as threshold instability, not distortion.')

# ── Incremental delta: how much CFM adds on top of Stage 1 ────────────────────
s1  = pop_table_base.groupby('marker')['pct_pos_norm'].mean().rename('stage1')
cfm = pop_table     .groupby('marker')['pct_pos_norm'].mean().rename('cfm')
raw = pop_table     .groupby('marker')['pct_pos_raw' ].mean().rename('raw')
incr = pd.concat([raw, s1, cfm], axis=1)
incr['s1_vs_raw']  = (incr['stage1'] - incr['raw']).round(1)
incr['cfm_vs_s1']  = (incr['cfm']    - incr['stage1']).round(1)
incr['cfm_vs_raw'] = (incr['cfm']    - incr['raw']).round(1)

print('\n── CFM Incremental Delta (Stage 2 vs Stage 1) ───────────────────────────')
print('   Large |delta vs raw| is Stage 1 correction. CFM adds the incremental column.')
print(f'   Mean |CFM incremental|: {incr["cfm_vs_s1"].abs().mean():.2f}%')
print()
print(incr[['raw','stage1','cfm','s1_vs_raw','cfm_vs_s1']].to_string(float_format='%.1f'))

# ── Secondary (reference only): reliability-filtered view ─────────────────────
# The density-at-threshold filter drops most unimodal markers — often leaving only
# ~3 markers, too few to compare methods. Kept for reference; NOT the headline metric.
print('\n' + '─' * 95)
print('Secondary (reference only) — reliability-filtered; drops most markers, not primary:')
print('─' * 95)
_ = summarize_positive_population(pop_table_base, min_reliable_frac=0.5, label='Stage 1 (base)')
_ = summarize_positive_population(pop_table,      min_reliable_frac=0.5, label='Stage 2 CFM (norm)')


In [ ]:
pivot = pop_table.pivot(index='marker', columns='sample', values='delta')
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(pivot.values, cmap='RdBu_r', vmin=-10, vmax=10, aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)
plt.colorbar(im, label='Δ % positive (norm − raw)')
ax.set_title('Positive Population Change: CFM Stage 2 vs Raw')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np, pandas as pd

S2_LABEL = 'Stage 2 CFM'

METHODS_DICT = {'Stage 1': pop_table_base, S2_LABEL: pop_table}

# --- Aggregate mean and SD of delta per marker ---
agg = {}
for method_name, pt in METHODS_DICT.items():
    g = pt.groupby('marker')['delta'].agg(['mean', 'std']).reset_index()
    g.columns = ['marker', 'mean_delta', 'std_delta']
    agg[method_name] = g

order        = list(adata.var_names)
method_names = list(METHODS_DICT.keys())
n_methods    = len(method_names)
n_markers    = len(order)

all_stds   = pd.concat([g['std_delta']        for g in agg.values()])
all_abs    = pd.concat([g['mean_delta'].abs() for g in agg.values()])
vmin, vmax = 0.0, float(all_stds.max())
size_scale = 800.0 / max(float(all_abs.max()), 1.0)
cmap       = plt.cm.coolwarm

fig, ax = plt.subplots(figsize=(2 + n_methods * 2, max(8, n_markers * 0.45)))

for x_idx, method_name in enumerate(method_names):
    g     = agg[method_name].set_index('marker').loc[order].reset_index()
    y_pos = np.arange(n_markers)
    sizes = np.abs(g['mean_delta'].values) * size_scale + 20
    ax.scatter(
        np.full(n_markers, x_idx), y_pos,
        s=sizes, c=g['std_delta'],
        cmap=cmap, vmin=vmin, vmax=vmax,
        edgecolors='k', linewidths=0.4, alpha=0.88,
    )

ax.set_xticks(range(n_methods))
ax.set_xticklabels(method_names, fontsize=11)
ax.set_yticks(range(n_markers))
ax.set_yticklabels(order, fontsize=9)
ax.set_xlim(-0.6, n_methods - 0.4)
ax.set_ylim(-0.7, n_markers - 0.3)
ax.invert_yaxis()
ax.grid(True, alpha=0.2)

for idx, mname in enumerate(order):
    if mname in bimodal_names:
        ax.get_yticklabels()[idx].set_color('navy')
        ax.get_yticklabels()[idx].set_fontweight('bold')

for ref_delta in [5, 10, 20]:
    ax.scatter([], [], s=ref_delta * size_scale + 20, c='gray', alpha=0.7,
               edgecolors='k', linewidths=0.4, label=f'{ref_delta}%')
ax.legend(title='|mean Δ|', loc='lower right', fontsize=8, title_fontsize=8, framealpha=0.9)

plt.colorbar(
    cm.ScalarMappable(norm=plt.Normalize(vmin, vmax), cmap=cmap),
    ax=ax,
    label='SD of Δ across samples\n(blue = consistent  ·  red = variable)',
    shrink=0.6,
)
ax.set_title(
    f'Positive Population Change — Stage 1 vs {S2_LABEL}\n'
    'Bubble size ∝ |mean Δ|  ·  Color = inter-sample SD  ·  Target: |Δ| < 5%\n'
    'Navy bold = bimodal markers',
    fontsize=11,
)
plt.tight_layout()
plt.show()

# --- Summary table ---
frames = []
for method_name, pt in METHODS_DICT.items():
    tmp = pt[['marker', 'delta']].copy()
    tmp['method'] = method_name
    frames.append(tmp)
all_delta = pd.concat(frames, ignore_index=True)

pivot = (
    all_delta.groupby(['marker', 'method'])['delta']
    .agg(['mean', 'std']).round(2)
    .unstack('method')
)
print('\nMean Δ (±SD) positive population per marker:')
print('=' * 70)
print(pivot.to_string())
print('\nTarget: |mean Δ| < 5% per marker')

In [ ]:
# === DETAILED PER-SAMPLE BREAKDOWN ===
# Show which samples are driving the large mean Δ values
# Identifies outlier samples and compares median vs mean

print('='*90)
print('DETAILED PER-SAMPLE BREAKDOWN — All Markers')
print('='*90)

S2_LABEL = 'Stage 2 CFM'
pop_tables = {'Stage 1': pop_table_base, S2_LABEL: pop_table}

for method_label, pt in pop_tables.items():
    print(f'\n{method_label} — Per-Sample Positive Population Delta\n')
    print('='*90)
    
    for marker in sorted(pt['marker'].unique()):
        marker_df = pt[pt['marker'] == marker].sort_values('delta')
        
        print(f'\n{marker}:')
        print('-'*90)
        print(marker_df[['sample', 'pct_pos_raw', 'pct_pos_norm', 'delta']].to_string(index=False))
        
        # Statistics
        mean_delta = marker_df['delta'].mean()
        median_delta = marker_df['delta'].median()
        std_delta = marker_df['delta'].std()
        min_delta = marker_df['delta'].min()
        max_delta = marker_df['delta'].max()
        
        within_5 = (abs(marker_df['delta']) <= 5).sum()
        total = len(marker_df)
        
        print(f'\nStats: Mean={mean_delta:+.2f}%, Median={median_delta:+.2f}%, '
              f'Std={std_delta:.2f}%, Range=[{min_delta:+.2f}%, {max_delta:+.2f}%]')
        print(f'Samples within ±5%: {within_5}/{total}')
        
        # Flag outliers
        outliers = marker_df[abs(marker_df['delta']) > 10]
        if len(outliers) > 0:
            print(f'⚠️  OUTLIERS (|Δ| > 10%):')
            for _, row in outliers.iterrows():
                print(f'    {row["sample"]}: {row["delta"]:+.2f}%')
    
    print(f'\n{method_label} SUMMARY:')
    print('-'*90)
    summary = pt.groupby('marker').agg({
        'delta': ['mean', 'median', 'std', 'min', 'max', 
                  lambda x: (abs(x) <= 5).sum()]
    }).round(2)
    summary.columns = ['mean', 'median', 'std', 'min', 'max', 'within_5_count']
    print(summary.to_string())
    
    print(f'\nTarget: |mean Δ| < 5% per marker\n')

In [ ]:
# ============================================================================
# 6b. SHAPE-PRESERVATION DIAGNOSTIC  (location-invariant)  — FAST version
# ----------------------------------------------------------------------------
# Positive-pop Δ (fraction over a threshold) and silhouette (3-blob 20D
# separation) are both SHAPE-BLIND: you can flatten a sharp peak into a broad
# smear and still pass both. This cell measures whether each marker's 1D
# marginal SHAPE survives, INDEPENDENT of any location shift Stage 1 / CFM apply.
#
# Per (marker, sample), in log1p space, normalized vs raw (1.0 = shape preserved):
#   peak_ratio = peak density(norm) / peak density(raw)   <1 => mode flattened
#   var_ratio  = var(norm) / var(raw)                     >1 => broadened
#   iqr_ratio  = IQR(norm) / IQR(raw)                     >1 => broadened (robust)
#
# peak_ratio uses SHARED bin edges (union range) so it is binning-fair.
# Stage 1 is a pure translation -> should sit ~1.0 everywhere (control);
# CFM is the test subject. ChromA's reported 175k->75k peak collapse should
# show up here as peak_ratio ~ 0.4 for CFM and ~1.0 for Stage 1.
#
# SPEED: per-sample row indices and log1p matrices are precomputed ONCE
# (the old version recomputed the sample mask + log1p inside a 2x20x18 loop
# over the full 1.76M-row array — that was the slow part).
# ============================================================================
from scipy.stats import iqr as _iqr

SHAPE_LAYERS = [('normalized_base', 'Stage 1'), ('normalized', 'Stage 2 CFM')]
N_SHAPE_BINS = 80
MIN_CELLS    = 50

# --- precompute ONCE ---
sample_ids_shape = adata_norm.obs['sample_id'].values
sample_unique    = np.unique(sample_ids_shape)
sample_idx       = {s: np.where(sample_ids_shape == s)[0] for s in sample_unique}
sample_idx       = {s: idx for s, idx in sample_idx.items() if idx.size >= MIN_CELLS}
X_raw_log        = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)


def _shape_ratios(raw_vals, norm_vals):
    lo = min(raw_vals.min(), norm_vals.min())
    hi = max(raw_vals.max(), norm_vals.max())
    if not np.isfinite(lo) or hi <= lo:
        return np.nan, np.nan, np.nan
    edges = np.linspace(lo, hi, N_SHAPE_BINS + 1)
    hr, _ = np.histogram(raw_vals,  bins=edges, density=True)
    hn, _ = np.histogram(norm_vals, bins=edges, density=True)
    peak = hn.max() / hr.max() if hr.max() > 0 else np.nan
    vr   = norm_vals.var() / raw_vals.var() if raw_vals.var() > 0 else np.nan
    ir_r = _iqr(raw_vals)
    ir   = _iqr(norm_vals) / ir_r if ir_r > 0 else np.nan
    return peak, vr, ir


rows = []
for layer_key, layer_label in SHAPE_LAYERS:
    X_layer_log = np.log1p(np.clip(np.asarray(adata_norm.layers[layer_key]), 0, None)).astype(np.float32)
    for mi, mname in enumerate(marker_names):
        raw_col  = X_raw_log[:, mi]
        norm_col = X_layer_log[:, mi]
        peaks, vrs, irs = [], [], []
        for idx in sample_idx.values():
            p, v, ir = _shape_ratios(raw_col[idx], norm_col[idx])
            if np.isfinite(p):  peaks.append(p)
            if np.isfinite(v):  vrs.append(v)
            if np.isfinite(ir): irs.append(ir)
        rows.append({
            'method': layer_label, 'marker': mname,
            'peak_ratio': np.mean(peaks) if peaks else np.nan,
            'var_ratio':  np.mean(vrs)  if vrs  else np.nan,
            'iqr_ratio':  np.mean(irs)  if irs  else np.nan,
        })
    del X_layer_log
shape_df = pd.DataFrame(rows)

# ---- side-by-side table: Stage 1 vs CFM per marker --------------------------
piv = shape_df.pivot(index='marker', columns='method',
                     values=['peak_ratio', 'var_ratio', 'iqr_ratio']).round(3)
piv = piv.reindex(columns=pd.MultiIndex.from_product(
    [['peak_ratio', 'var_ratio', 'iqr_ratio'], ['Stage 1', 'Stage 2 CFM']]))
piv = piv.sort_values(('peak_ratio', 'Stage 2 CFM'))

print('=' * 92)
print('SHAPE-PRESERVATION DIAGNOSTIC  (1.0 = shape preserved; per-sample mean vs raw, log1p)')
print('  peak_ratio <1 = mode flattened   |   var_ratio / iqr_ratio >1 = broadened')
print('=' * 92)
print(piv.to_string())

# ---- flag markers CFM distorts ---------------------------------------------
cfm = shape_df[shape_df.method == 'Stage 2 CFM'].set_index('marker')
s1  = shape_df[shape_df.method == 'Stage 1'].set_index('marker')
PEAK_LO, VAR_HI, VAR_LO = 0.80, 1.25, 0.80
distorted = cfm[(cfm.peak_ratio < PEAK_LO) |
                (cfm.var_ratio > VAR_HI) | (cfm.var_ratio < VAR_LO)]
print('\nCFM SHAPE-DISTORTED markers '
      f'(peak<{PEAK_LO} or var outside [{VAR_LO},{VAR_HI}]):')
if len(distorted) == 0:
    print('  none — CFM preserves 1D shape on all markers')
else:
    for m, r in distorted.sort_values('peak_ratio').iterrows():
        print(f'  {m:>10s}  peak={r.peak_ratio:.2f} (S1 {s1.loc[m,"peak_ratio"]:.2f})  '
              f'var={r.var_ratio:.2f} (S1 {s1.loc[m,"var_ratio"]:.2f})  '
              f'iqr={r.iqr_ratio:.2f}')

print(f'\nMean peak_ratio  — Stage 1: {s1.peak_ratio.mean():.3f}   '
      f'CFM: {cfm.peak_ratio.mean():.3f}')
print(f'Mean var_ratio   — Stage 1: {s1.var_ratio.mean():.3f}   '
      f'CFM: {cfm.var_ratio.mean():.3f}')

# ---- bar charts: peak_ratio and var_ratio, Stage 1 vs CFM ------------------
order = list(piv.index)
x = np.arange(len(order)); w = 0.4
fig, axes = plt.subplots(1, 2, figsize=(20, 6))
for ax, ratio, ref_lines in [
    (axes[0], 'peak_ratio', [1.0]),
    (axes[1], 'var_ratio',  [1.0, VAR_HI, VAR_LO]),
]:
    ax.bar(x - w/2, [s1.loc[m, ratio]  for m in order], w, label='Stage 1',     color='steelblue',   alpha=0.85)
    ax.bar(x + w/2, [cfm.loc[m, ratio] for m in order], w, label='Stage 2 CFM', color='forestgreen', alpha=0.85)
    for rl in ref_lines:
        ax.axhline(rl, color='red', linestyle='--', alpha=0.4)
    ax.set_xticks(x); ax.set_xticklabels(order, rotation=45, ha='right', fontsize=9)
    ax.set_title(f'{ratio}  (1.0 = preserved)'); ax.set_ylabel(ratio)
    ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.suptitle('Shape preservation: Stage 1 (pure shift, control) vs CFM (transport)', fontsize=13)
plt.tight_layout(); plt.show()

print('\nReading it: Stage 1 bars hug 1.0 (translation preserves shape). Where CFM bars')
print('drop below 1.0 on peak_ratio (and rise above on var_ratio), CFM is smearing that')
print("marker's distribution — the cost that positive-pop Δ and silhouette cannot see.")


## 7. Histogram Comparison PDF

Per-sample histogram PDFs for all markers. Saved to `histograms_cfm/`.

In [ ]:
import os
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import colormaps

save_dir = 'histograms_cfm'
os.makedirs(save_dir, exist_ok=True)

layers_to_plot = [
    ('normalized_base', 'Stage 1 (analytic)'),
    ('normalized',      f'Stage 2 (OT-CFM, n_steps={N_STEPS})'),
]

sample_col = 'sample_id' if 'sample_id' in adata_norm.obs.columns else 'batch_id'
sample_ids = adata_norm.obs[sample_col].astype(str).values
unique_samples = sorted(np.unique(sample_ids).tolist())
cmap = colormaps.get_cmap('tab20')
colors = {s: cmap(i % 20) for i, s in enumerate(unique_samples)}

pdf_path = os.path.join(save_dir, 'cfm_histograms.pdf')
rows_per_page = 4

# FIXED-BIN PATCH: per marker, derive ONE set of bin edges from the union of both
# layers' log1p range, and use it for every layer/sample. With per-layer-auto bins
# (the old default) a wider range gives wider bins -> higher peak counts, so peak
# heights were NOT comparable across the Stage 1 vs CFM columns. Shared edges +
# sharey='row' make the peak-height change (e.g. ChromA 175k -> 75k) a real,
# directly-readable comparison rather than a binning artifact.
def _marker_edges(m_idx, n_bins=80):
    lo, hi = np.inf, -np.inf
    for layer_key, _ in layers_to_plot:
        col = np.asarray(adata_norm.layers[layer_key][:, m_idx]).ravel()
        col = np.log1p(np.clip(col, 0, None))
        lo, hi = min(lo, float(col.min())), max(hi, float(col.max()))
    if not np.isfinite(lo) or hi <= lo:
        hi = lo + 1.0
    return np.linspace(lo, hi, n_bins + 1)

with PdfPages(pdf_path) as pdf:
    page_markers = []
    for i, mname in enumerate(marker_names):
        page_markers.append(mname)
        if len(page_markers) == rows_per_page or i == len(marker_names) - 1:
            fig, axes = plt.subplots(len(page_markers), len(layers_to_plot),
                                     figsize=(7 * len(layers_to_plot), 4 * len(page_markers)),
                                     sharey='row')  # same y per marker -> peaks comparable
            if len(page_markers) == 1:
                axes = axes.reshape(1, -1)
            for row, mname in enumerate(page_markers):
                m_idx = marker_names.index(mname)
                edges = _marker_edges(m_idx)               # shared across both columns
                centers = 0.5 * (edges[:-1] + edges[1:])
                for col, (layer_key, label) in enumerate(layers_to_plot):
                    ax = axes[row, col]
                    X_col = np.asarray(adata_norm.layers[layer_key][:, m_idx]).ravel()
                    X_log = np.log1p(np.clip(X_col, 0, None))
                    for s in unique_samples:
                        mask = sample_ids == s
                        c, _ = np.histogram(X_log[mask], bins=edges)   # SHARED edges
                        ax.plot(centers, c, color=colors[s], linewidth=1.5, alpha=0.7, label=s)
                    bim_tag = ' [BIMODAL]' if marker_is_bimodal[m_idx] else ''
                    ax.set_title(f'{mname}{bim_tag} — {label}', fontsize=9)
                    ax.set_xlabel('log1p intensity', fontsize=8)
                    if col == 0:
                        ax.set_ylabel('Count', fontsize=8)
                    if row == 0 and col == 0:
                        ax.legend(fontsize=5, ncol=3, frameon=False)
                    ax.spines[['top', 'right']].set_visible(False)
            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)
            page_markers = []

print(f'PDF saved to: {pdf_path}')
print('Bins are now shared per marker across both columns, and y-axis is shared per row,')
print('so a peak-height drop (e.g. ChromA) is a real shape change, not a binning artifact.')


## 8. kBET Evaluation

kBET acceptance rate for 5 clinical groups pairing samples from different batches.

- **Higher kBET = better** (batches well-mixed in local neighborhoods)
- Stage 1 baseline: ~0.631
- UniFORM baseline: 0.631
- Target: Stage 2 CFM > 0.631 with positive population |Δ| < 5%

In [ ]:
import pegasus as pg
import pegasusio as pgio

def subset_ab(adata, sample, batch):
    mask = (adata.obs['sample_id'] == sample) & (adata.obs['batch_id'] == batch)
    return adata[mask, :].copy()

GROUP_DEFS = {
    'g1': [('PRAD-02', 'batch1'), ('PRAD-14', 'batch7')],
    'g2': [('PRAD-01', 'batch4'), ('PRAD-02', 'batch1')],
    'g3': [('PRAD-05', 'batch1'), ('PRAD-12', 'batch2')],
    'g4': [('PRAD-07', 'batch2'), ('PRAD-19', 'batch6')],
    'g5': [('PRAD-02', 'batch1'), ('PRAD-07', 'batch2')],
}

EVAL_LAYERS = ['normalized_base', 'normalized']
print(f'Evaluating: {EVAL_LAYERS}')
print()
print('Baselines:')
print('  Stage 1 (analytic):              ~0.631')
print('  UniFORM:                          0.631')
print('  SpaNCy-GNN ensemble hybrid:       0.574')
print(f'  Stage 2 CFM (n_steps={N_STEPS}):   ???  (target > 0.631)')

In [ ]:
all_kbet = {}
umap_coords = {}

for layer_name in EVAL_LAYERS:
    print(f'\n{"-"*55}\n  {layer_name}\n{"-"*55}')
    adata_kbet = adata_norm.copy()
    adata_kbet.X = adata_norm.layers[layer_name].copy()

    layer_res = {}
    for gname, pairs in GROUP_DEFS.items():
        try:
            parts = [subset_ab(adata_kbet, s, b) for s, b in pairs]
            adata_g = parts[0].concatenate(parts[1], batch_key=None)
            mmdata = pgio.MultimodalData(adata_g)
            pg.qc_metrics(mmdata)
            pg.filter_data(mmdata)
            pg.identify_robust_genes(mmdata)
            pg.pca(mmdata, features=None, n_components=20)
            pg.neighbors(mmdata)
            pg.umap(mmdata)
            umap_coords.setdefault(layer_name, {})[gname] = (
                mmdata.obsm['X_umap'].copy(),
                mmdata.obs['batch_id'].values.copy(),
            )
            stat, pval, score = pg.calc_kBET(mmdata, attr='batch_id', rep='umap')
            layer_res[gname] = {'kBET': score, 'chi2': stat, 'p': pval}
            print(f'  {gname}: kBET={score:.4f}  chi2={stat:.4f}  p={pval:.4f}')
        except Exception as e:
            print(f'  {gname}: FAILED — {e}')
            layer_res[gname] = {'kBET': float('nan'), 'chi2': float('nan'), 'p': float('nan')}

    all_kbet[layer_name] = layer_res

print('\nDone.')

In [ ]:
print('=' * 70)
print('kBET Summary (mean across groups — higher = better)')
print('=' * 70)

summary_rows = []
for layer_name, res in all_kbet.items():
    kbets = [v['kBET'] for v in res.values() if not np.isnan(v['kBET'])]
    chi2s = [v['chi2'] for v in res.values() if not np.isnan(v['chi2'])]
    if kbets:
        row = {
            'layer': layer_name,
            'mean_kBET': float(np.mean(kbets)),
            'mean_chi2': float(np.mean(chi2s)),
            'n_groups': f'{len(kbets)}/{len(res)}',
        }
        summary_rows.append(row)
        print(f"  {layer_name:>20s}  kBET={row['mean_kBET']:.4f}  "
              f"chi2={row['mean_chi2']:.4f}  ({row['n_groups']} groups)")

print()
for gname, res in all_kbet.get('normalized', {}).items():
    if not np.isnan(res['kBET']):
        print(f"{gname}: kBET={res['kBET']:.4f}  chi2={res['chi2']:.4f}  p={res['p']:.4f}")

print()
print('Baselines:')
print('  Stage 1 expected:            kBET~0.631')
print('  UniFORM:                     kBET=0.631')
print(f'  Stage 2 CFM (n_steps={N_STEPS}):  kBET=???  (target > 0.631)')

df_kbet = pd.DataFrame(summary_rows)

if len(df_kbet) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    colors = ['steelblue', 'forestgreen'][:len(df_kbet)]
    axes[0].bar(df_kbet['layer'], df_kbet['mean_kBET'], color=colors, alpha=0.85)
    axes[0].axhline(0.631, color='red', linestyle='--', linewidth=1.5, label='UniFORM 0.631')
    axes[0].set_ylabel('Mean kBET acceptance rate')
    axes[0].set_title('kBET (higher = better)')
    axes[0].set_xticklabels(df_kbet['layer'], rotation=20, ha='right', fontsize=9)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')
    axes[1].bar(df_kbet['layer'], df_kbet['mean_chi2'], color=colors, alpha=0.85)
    axes[1].set_ylabel('Mean chi-square')
    axes[1].set_title('Chi-square (lower = better)')
    axes[1].set_xticklabels(df_kbet['layer'], rotation=20, ha='right', fontsize=9)
    axes[1].grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

In [ ]:
# Silhouette score — UniFORM-style: 3 cell types, computed PER SAMPLE
# Tumor epithelial (ECAD+), immune (CD45+), non-immune stromal (aSMA+)
#
# Why per-sample (not per clinical group): computing silhouette across two
# batch-paired samples lets the cross-batch offset inflate the raw baseline —
# good normalization then looks like it *lowers* silhouette (it just removed the
# batch-driven separation). Computing within each sample removes that confound:
# the score reflects only whether the 3 cell populations stay distinct.
# Labels are assigned once on raw data, fixed across all methods.

N_SIL = 3000  # cells per sample; O(n²) — stable well below this

# Step 1: GMM thresholds for 3 anchor markers on full raw dataset (log1p space)
anchor_names = ['ECAD', 'CD45', 'aSMA']
anchor_thresholds = {}
print('Anchor marker GMM thresholds (log1p, raw data):')
for mname in anchor_names:
    mi = marker_names.index(mname)
    gmm = GaussianMixture(n_components=2, random_state=0).fit(
        X_log_preview[:, mi].reshape(-1, 1)
    )
    thr = float(np.mean(gmm.means_))
    anchor_thresholds[mname] = thr
    print(f'  {mname}: {thr:.3f}')

# Step 2: assign cell type labels on raw data — fixed for all methods
#   0 = tumor epithelial (ECAD+)
#   1 = immune           (CD45+)
#   2 = stromal          (aSMA+)
#  -1 = unassigned
# Priority for conflicts: ECAD > CD45 > aSMA
ecad_pos = X_log_preview[:, marker_names.index('ECAD')] > anchor_thresholds['ECAD']
cd45_pos = X_log_preview[:, marker_names.index('CD45')] > anchor_thresholds['CD45']
asma_pos = X_log_preview[:, marker_names.index('aSMA')] > anchor_thresholds['aSMA']

cell_type = np.full(len(X_log_preview), -1, dtype=np.int8)
cell_type[asma_pos] = 2
cell_type[cd45_pos] = 1
cell_type[ecad_pos] = 0

print('\nCell type assignments (raw data):')
for code, label in [(0, 'tumor epithelial (ECAD+)'),
                    (1, 'immune (CD45+)'),
                    (2, 'stromal (aSMA+)'),
                    (-1, 'unassigned')]:
    n = (cell_type == code).sum()
    print(f'  {label}: {n:,}  ({100*n/len(cell_type):.1f}%)')

# Step 3: silhouette per sample, per method (no cross-batch mixing)
print(f'\nSilhouette score per sample — 3 cell types in 20D log1p space (higher = better):')
sample_ids_all = adata_norm.obs['sample_id'].values
sil_rows = []
for sample in sorted(np.unique(sample_ids_all)):
    smask = sample_ids_all == sample
    s_labels = cell_type[smask]
    valid = np.where(s_labels >= 0)[0]
    if len(valid) < 15:
        continue

    rng = np.random.default_rng(42)
    sel = valid[rng.choice(len(valid), min(N_SIL, len(valid)), replace=False)]
    sel_labels = s_labels[sel]
    classes = np.unique(sel_labels)
    if len(classes) < 2 or any((sel_labels == c).sum() < 5 for c in classes):
        continue  # need ≥2 cell types with ≥5 cells each

    sil_by_layer = {}
    for layer_name in ['raw'] + EVAL_LAYERS:
        if layer_name == 'raw':
            Xs = X_raw[smask]
        else:
            Xs = np.asarray(adata_norm.layers[layer_name])[smask]
        X_log = np.log1p(np.clip(Xs[sel], 0, None))
        sil_by_layer[layer_name] = round(
            float(silhouette_score(X_log, sel_labels, metric='euclidean')), 4
        )
    sil_rows.append({'sample': sample, **sil_by_layer})

sil_df = pd.DataFrame(sil_rows).set_index('sample')
sil_df.loc['Mean'] = sil_df.mean().round(4)
print(sil_df.to_string())
print(f'\n({len(sil_rows)} samples passed the ≥2-cell-type guard)')
print('(higher = better — tumor / immune / stromal populations stay distinct after normalization)')


In [ ]:
VIZ_LAYER = 'normalized'
palette = plt.cm.tab10.colors
fig, axes = plt.subplots(5, 2, figsize=(12, 28))
fig.suptitle('Batch mixing per clinical group — Raw vs Normalized', fontsize=14, y=1.01)

for row_idx, (gname, pairs) in enumerate(GROUP_DEFS.items()):
    parts = [subset_ab(adata_norm, s, b) for s, b in pairs]
    adata_g = parts[0].concatenate(parts[1], batch_key=None)
    unique_b = sorted(adata_g.obs['batch_id'].unique())

    # Raw UMAP — compute via pegasus (same pipeline as kBET)
    umap_raw, batch_raw = None, None
    try:
        mmdata_raw = pgio.MultimodalData(adata_g.copy())
        pg.qc_metrics(mmdata_raw); pg.filter_data(mmdata_raw)
        pg.identify_robust_genes(mmdata_raw)
        pg.pca(mmdata_raw, features=None, n_components=20)
        pg.neighbors(mmdata_raw); pg.umap(mmdata_raw)
        umap_raw  = mmdata_raw.obsm['X_umap']
        batch_raw = mmdata_raw.obs['batch_id'].values
    except Exception as e:
        print(f'{gname} raw UMAP failed: {e}')

    # Normalized UMAP — reuse from kBET loop (no recomputation)
    umap_nrm, batch_nrm = None, None
    if VIZ_LAYER in umap_coords and gname in umap_coords[VIZ_LAYER]:
        umap_nrm, batch_nrm = umap_coords[VIZ_LAYER][gname]

    for col_idx, (coords, batch_ids, title) in enumerate([
            (umap_raw, batch_raw, 'Raw'),
            (umap_nrm, batch_nrm, f'Normalized ({VIZ_LAYER})'),
    ]):
        ax = axes[row_idx, col_idx]
        if coords is None:
            ax.text(0.5, 0.5, 'UMAP unavailable', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{gname} — {title}', fontsize=10)
            continue
        for ci, b in enumerate(unique_b):
            mask = batch_ids == b
            ax.scatter(coords[mask, 0], coords[mask, 1],
                       c=[palette[ci % len(palette)]], s=2, alpha=0.3, rasterized=True, label=b)
        ax.set_title(f'{gname} — {title}', fontsize=10)
        ax.set_xlabel('UMAP 1', fontsize=8)
        ax.set_ylabel('UMAP 2', fontsize=8)
        ax.tick_params(labelsize=7)
        if col_idx == 1:
            ax.legend(markerscale=4, fontsize=8, loc='upper right',
                      title='batch_id', title_fontsize=8)

plt.tight_layout()
plt.savefig('umap_batch_mixing.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: umap_batch_mixing.png')